In [3]:
import os
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq.chat_models import ChatGroq
import requests
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


llm= ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    api_key= os.getenv("GROQ_API_KEY")
)

In [ ]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. "AAPL", "TSLA")
    using Alpha Vantage with API key in the URL.
    """
    url= f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={os.getenv('STOCK_API_KEY')}"
    r= requests.get(url)
    return r.json()

@tool
def purchase_stock(symbol:str, amount)-> dict:
    """
    Simulate purchasing a given quantity of stock symbol.

    HUMAN-IN-THE-LOOP
    Before confirming the purchase, this tool will interrupt
    and wait for human decision ("yes"/ anything else).
    """

    # this pauses the graph and return control to the caller
    decision= interrupt(f"Approve buying {amount} shares of {symbol}? (yes/no)")

    if isinstance(decision, str) and decision.lower() =='yes':
        return {
            'status': 'success',
            "message":f"Purchase order placed for {amount} shares of {symbol}",
            'symbol':symbol,
            'quantity':amount
        }
    
    else:
        return {
            'status': 'cancelled',
            "message":f"Purchase order placed for {amount} shares of {symbol} was declined by human.",
            'symbol':symbol,
            'quantity':amount
        }

tools_list= [get_stock_price, purchase_stock]
tool_node= ToolNode(tools_list)

llm_with_tools= llm.bind_tools(tools_list)

In [21]:
def chat_point(state:ChatState):
    messages= [SystemMessage(content="You are a QUESTION-ANSWER assistent.And answer only if you know other-wise explicitly tell you don't know!"), *state['messages']]

    result= llm_with_tools.invoke(messages)

    return {"messages":[result]}


In [23]:
checkpointer= InMemorySaver()

builder= StateGraph(ChatState)

builder.add_node('chat_node', chat_point)
builder.add_node('tools', tool_node)

builder.add_edge(START, 'chat_node')
builder.add_conditional_edges('chat_node', tools_condition)
builder.add_edge('tools', "chat_node")

graph= builder.compile(checkpointer)

CONFIG= {'configurable':{'thread_id':1}}

user_query= input("User: ")
while user_query.strip().lower() not in ['stop', 'exit']:
    print(f"User: {user_query}")

    result= graph.invoke({"messages":[HumanMessage(content=user_query)]}, config=CONFIG)

    if '__interrupt__' in result.keys():
        print(f"Tool: {result['__interrupt__'][0].value}")
        user_approval_status= input(result['__interrupt__'][0].value)
        final_result= graph.invoke(
            Command(resume=user_approval_status),
            config=CONFIG
        )
        print(f"Ai: {final_result['messages'][-1].content}")

    else:
        print(f"Ai: {result['messages'][-1].content}")

    user_query= input("User: ")




User: what is stock price of AAPL
Ai: The current stock price of AAPL is $316.22.
User: buy 6 stock of AAPL
Tool: Approve buying 6 shares of AAPL? (yes/no)
yes
Ai: The purchase order has been placed for 6 shares of AAPL.
